# 02 — Embedding, perbandingan pencocokan, dan robustness

Salinan publik dari alur eksperimen. Tidak memuat dataset, nama peserta, embedding, foto, output cell, atau path komputer penulis. Gunakan dataset sendiri yang berhak diproses. Hasil numerik akan bergantung pada dataset dan tidak harus sama dengan README.

Setelah dijalankan pada data pribadi, output notebook dapat kembali mengandung nama atau wajah. Jalankan script pembersih sebelum commit. Lihat `notebooks_public/README.md` untuk prasyarat dan batas reproduksi.

Selesaikan notebook 01 dahulu. Setiap identitas memerlukan 3–4 foto untuk protokol ini. Dua referensi dan satu query dipilih per percobaan; bila ada foto keempat, foto tersebut dapat terpilih pada percobaan lain. Transformasi robustness diterapkan setelah alignment dan galeri tetap normal.

## Encoder dan input

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from insightface import model_zoo

from pathlib import Path
import sys

def find_project_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("Jalankan notebook di dalam checkout repository proyek.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


METADATA_DIR = PROJECT_ROOT / "dataset" / "metadata"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = (
    PROJECT_ROOT
    / "weights"
    / "insightface"
    / "models"
    / "buffalo_l"
    / "w600k_r50.onnx"
)

if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Model tidak ditemukan: {MODEL_PATH}")

alignment_df = pd.read_csv(
    METADATA_DIR / "alignment_manifest.csv"
)

embedding_input_df = (
    alignment_df.loc[
        alignment_df["alignment_status"].eq("success")
    ]
    .sort_values("image_id")
    .reset_index(drop=True)
)

if embedding_input_df.empty:
    raise ValueError("Belum ada foto yang berhasil di-alignment.")

if embedding_input_df["image_id"].duplicated().any():
    raise ValueError("Ada image_id duplikat.")

encoder = model_zoo.get_model(
    str(MODEL_PATH),
    providers=["CPUExecutionProvider"]
)

if encoder is None or encoder.taskname != "recognition":
    raise ValueError("File tidak dikenali sebagai encoder wajah.")

encoder.prepare(ctx_id=-1)

print("Encoder siap.")
print("Ukuran input    :", encoder.input_size)
print("Foto diproses   :", len(embedding_input_df))
print("Jumlah identitas:", embedding_input_df["person_id"].nunique())

## Ekstraksi embedding

In [ ]:
import time

from PIL import Image
from tqdm.auto import tqdm

vectors = []
metadata_rows = []
error_rows = []

started = time.perf_counter()

for record in tqdm(
    embedding_input_df.to_dict("records"),
    desc="Ekstraksi embedding",
    unit="foto"
):
    try:
        image_path = PROJECT_ROOT / record["aligned_path"]

        with Image.open(image_path) as source:
            if source.size != tuple(encoder.input_size):
                raise ValueError(
                    f"Ukuran gambar tidak sesuai: {source.size}"
                )

            # Encoder menerima urutan warna BGR.
            image_bgr = np.asarray(
                source.convert("RGB")
            )[:, :, ::-1].copy()

        vector = np.asarray(
            encoder.get_feat(image_bgr),
            dtype=np.float32
        ).reshape(-1)

        if vector.size != 512:
            raise ValueError(
                f"Dimensi embedding tidak sesuai: {vector.size}"
            )

        if not np.isfinite(vector).all():
            raise ValueError("Embedding mengandung NaN atau infinity.")

        vector_norm = float(np.linalg.norm(vector))

        if vector_norm <= 0:
            raise ValueError("Embedding memiliki panjang nol.")

        # Normalisasi L2 untuk pencocokan cosine similarity.
        vector = vector / vector_norm

        vectors.append(vector)

        metadata_rows.append({
            "image_id": record["image_id"],
            "person_id": record["person_id"],
            "person_name": record["person_name"],
            "aligned_path": record["aligned_path"],
        })

    except Exception as exc:
        error_rows.append({
            "image_id": record["image_id"],
            "error": f"{type(exc).__name__}: {exc}",
        })

print(
    f"\nWaktu ekstraksi: "
    f"{time.perf_counter() - started:.1f} detik"
)
print("Berhasil:", len(vectors))
print("Gagal   :", len(error_rows))

if error_rows:
    display(pd.DataFrame(error_rows))
    raise ValueError(
        "Selesaikan kegagalan sebelum menyimpan embedding final."
    )

embeddings = np.stack(vectors).astype(np.float32)
embedding_metadata_df = pd.DataFrame(metadata_rows)

embedding_path = ARTIFACTS_DIR / "arcface_embeddings.npz"

np.savez_compressed(
    embedding_path,
    embeddings=embeddings,
    image_ids=embedding_metadata_df["image_id"].to_numpy(dtype=str),
    person_ids=embedding_metadata_df["person_id"].to_numpy(dtype=str)
)

embedding_metadata_df.to_csv(
    ARTIFACTS_DIR / "arcface_embedding_metadata.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Embedding tersimpan:", embedding_path)

## Validasi embedding

In [ ]:
with np.load(
    ARTIFACTS_DIR / "arcface_embeddings.npz",
    allow_pickle=False
) as saved:
    X = saved["embeddings"]
    image_ids = saved["image_ids"]
    person_ids = saved["person_ids"]

lengths = np.linalg.norm(X, axis=1)

print("Bentuk embedding :", X.shape)
print("Jumlah image_id  :", len(np.unique(image_ids)))
print("Jumlah identitas :", len(np.unique(person_ids)))
print("Seluruh angka valid:", np.isfinite(X).all())
print(
    "Normalisasi valid:",
    np.allclose(lengths, 1.0, atol=1e-5)
)

## Metadata dan label

In [ ]:
import numpy as np
import pandas as pd

with np.load(
    ARTIFACTS_DIR / "arcface_embeddings.npz",
    allow_pickle=False
) as saved:
    X = saved["embeddings"]
    image_ids = saved["image_ids"]
    person_ids = saved["person_ids"]

metadata_df = pd.read_csv(
    ARTIFACTS_DIR / "arcface_embedding_metadata.csv"
)

if metadata_df["image_id"].duplicated().any():
    raise ValueError("Metadata memiliki image_id duplikat.")

if set(metadata_df["image_id"]) != set(image_ids):
    raise ValueError("Metadata dan embedding tidak cocok.")

# Susun metadata mengikuti urutan embedding.
metadata_df = (
    metadata_df
    .set_index("image_id")
    .loc[image_ids]
    .reset_index()
)

if not np.array_equal(
    metadata_df["person_id"].to_numpy(),
    person_ids
):
    raise ValueError("Label metadata dan embedding berbeda.")

if (
    not np.isfinite(X).all()
    or not np.allclose(
        np.linalg.norm(X, axis=1), 1.0, atol=1e-5
    )
):
    raise ValueError("Embedding tidak valid atau belum dinormalisasi.")

classes, class_indices = np.unique(
    person_ids,
    return_inverse=True
)

class_counts = np.bincount(class_indices)

if class_counts.min() < 2:
    raise ValueError("Setiap identitas memerlukan minimal dua foto.")

print("Foto      :", len(X))
print("Identitas :", len(classes))
print("Foto per orang:")
print(pd.Series(class_counts).value_counts().sort_index())

## Leave-one-image-out

In [ ]:
from tqdm.auto import tqdm

# Karena embedding sudah dinormalisasi,
# dot product sama dengan cosine similarity.
similarity_matrix = X @ X.T

# Foto tidak boleh dicocokkan dengan dirinya sendiri.
np.fill_diagonal(similarity_matrix, -np.inf)

class_sums = np.zeros(
    (len(classes), X.shape[1]),
    dtype=np.float32
)

np.add.at(class_sums, class_indices, X)

prediction_rows = []

for query_index in tqdm(
    range(len(X)),
    desc="Evaluasi leave-one-image-out",
    unit="foto"
):
    true_person_id = person_ids[query_index]
    own_class_index = class_indices[query_index]

    # Metode 1: foto galeri terdekat.
    nearest_index = int(
        np.argmax(similarity_matrix[query_index])
    )

    prediction_rows.append({
        "method": "nearest_template",
        "query_image_id": image_ids[query_index],
        "true_person_id": true_person_id,
        "predicted_person_id": person_ids[nearest_index],
        "similarity": float(
            similarity_matrix[query_index, nearest_index]
        ),
        "matched_image_id": image_ids[nearest_index],
        "own_identity_gallery_size": int(
            class_counts[own_class_index] - 1
        ),
    })

    # Metode 2: centroid setiap identitas.
    # Keluarkan foto uji dari centroid identitasnya.
    gallery_sums = class_sums.copy()
    gallery_counts = class_counts.copy()

    gallery_sums[own_class_index] -= X[query_index]
    gallery_counts[own_class_index] -= 1

    centroids = (
        gallery_sums / gallery_counts[:, None]
    )

    centroid_norms = np.linalg.norm(
        centroids,
        axis=1,
        keepdims=True
    )

    if np.any(centroid_norms <= 1e-12):
        raise ValueError("Ditemukan centroid dengan panjang nol.")

    centroids = centroids / centroid_norms

    centroid_scores = centroids @ X[query_index]
    best_class_index = int(np.argmax(centroid_scores))

    prediction_rows.append({
        "method": "centroid",
        "query_image_id": image_ids[query_index],
        "true_person_id": true_person_id,
        "predicted_person_id": classes[best_class_index],
        "similarity": float(
            centroid_scores[best_class_index]
        ),
        "matched_image_id": None,
        "own_identity_gallery_size": int(
            gallery_counts[own_class_index]
        ),
    })

predictions_df = pd.DataFrame(prediction_rows)

predictions_df["correct"] = (
    predictions_df["true_person_id"]
    == predictions_df["predicted_person_id"]
)

print("Evaluasi selesai.")

## Metrik leave-one-image-out

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

summary_rows = []

for method, group in predictions_df.groupby("method"):
    summary_rows.append({
        "method": method,
        "n_queries": len(group),
        "correct": int(group["correct"].sum()),
        "incorrect": int((~group["correct"]).sum()),
        "accuracy": accuracy_score(
            group["true_person_id"],
            group["predicted_person_id"]
        ),
        "macro_f1": f1_score(
            group["true_person_id"],
            group["predicted_person_id"],
            labels=classes,
            average="macro",
            zero_division=0
        ),
        "mean_person_accuracy": (
            group.groupby("true_person_id")["correct"]
            .mean()
            .mean()
        ),
    })

summary_df = pd.DataFrame(summary_rows)

REPORT_DIR = PROJECT_ROOT / "reports" / "experiments"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

predictions_df.to_csv(
    REPORT_DIR / "arcface_loo_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

summary_df.to_csv(
    REPORT_DIR / "arcface_loo_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

display(summary_df.round(4))

print("Prediksi yang salah:")
display(
    predictions_df.loc[
        ~predictions_df["correct"],
        [
            "method",
            "query_image_id",
            "true_person_id",
            "predicted_person_id",
            "similarity"
        ]
    ]
)

## Pembagian few-shot

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score

from pathlib import Path
import sys

def find_project_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("Jalankan notebook di dalam checkout repository proyek.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
METADATA_DIR = PROJECT_ROOT / "dataset" / "metadata"
REPORT_DIR = PROJECT_ROOT / "reports" / "experiments"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

with np.load(
    ARTIFACTS_DIR / "arcface_embeddings.npz",
    allow_pickle=False
) as saved:
    X = saved["embeddings"]
    image_ids = saved["image_ids"]
    person_ids = saved["person_ids"]

if len(np.unique(image_ids)) != len(image_ids):
    raise ValueError("Ada image_id duplikat.")

if (
    not np.isfinite(X).all()
    or not np.allclose(
        np.linalg.norm(X, axis=1), 1.0, atol=1e-5
    )
):
    raise ValueError("Embedding tidak valid atau belum dinormalisasi.")

classes = np.unique(person_ids)

indices_by_person = {
    person_id: np.flatnonzero(person_ids == person_id)
    for person_id in classes
}

if min(map(len, indices_by_person.values())) < 3:
    raise ValueError(
        "Setiap orang perlu minimal tiga foto untuk evaluasi ini."
    )

SEEDS = list(range(42, 72))

print("Jumlah identitas:", len(classes))
print("Jumlah percobaan:", len(SEEDS))

## Evaluasi 30 pembagian

In [ ]:
prediction_frames = []
manifest_rows = []

for trial, seed in enumerate(
    tqdm(SEEDS, desc="Evaluasi few-shot", unit="percobaan"),
    start=1
):
    rng = np.random.default_rng(seed)

    query_indices = []
    gallery_pairs = []

    for person_id in classes:
        shuffled = rng.permutation(
            indices_by_person[person_id]
        )

        query_index = int(shuffled[0])
        gallery_pair = shuffled[1:3]

        query_indices.append(query_index)
        gallery_pairs.append(gallery_pair)

        manifest_rows.append({
            "trial": trial,
            "seed": seed,
            "person_id": person_id,
            "query_image_id": image_ids[query_index],
            "gallery_image_id_1": image_ids[gallery_pair[0]],
            "gallery_image_id_2": image_ids[gallery_pair[1]],
        })

    query_indices = np.asarray(query_indices)
    gallery_pairs = np.asarray(gallery_pairs)

    # Pastikan foto uji tidak masuk galeri.
    if np.intersect1d(
        query_indices,
        gallery_pairs.ravel()
    ).size:
        raise ValueError("Foto uji ditemukan dalam galeri.")

    query_vectors = X[query_indices]
    true_labels = person_ids[query_indices]

    for n_shots in [1, 2]:
        gallery_indices = gallery_pairs[:, :n_shots]

        # Metode nearest template.
        flat_indices = gallery_indices.ravel()
        gallery_vectors = X[flat_indices]
        gallery_labels = person_ids[flat_indices]

        template_scores = query_vectors @ gallery_vectors.T
        best_template = np.argmax(template_scores, axis=1)

        template_predictions = gallery_labels[best_template]
        template_best_scores = template_scores[
            np.arange(len(query_indices)),
            best_template
        ]

        # Metode centroid.
        centroids = X[gallery_indices].mean(axis=1)

        lengths = np.linalg.norm(
            centroids,
            axis=1,
            keepdims=True
        )

        if np.any(lengths <= 1e-12):
            raise ValueError("Ditemukan centroid dengan panjang nol.")

        centroids = centroids / lengths

        centroid_scores = query_vectors @ centroids.T
        best_centroid = np.argmax(centroid_scores, axis=1)

        centroid_predictions = classes[best_centroid]
        centroid_best_scores = centroid_scores[
            np.arange(len(query_indices)),
            best_centroid
        ]

        outputs = [
            (
                "nearest_template",
                template_predictions,
                template_best_scores
            ),
            (
                "centroid",
                centroid_predictions,
                centroid_best_scores
            ),
        ]

        for method, predictions, similarities in outputs:
            prediction_frames.append(
                pd.DataFrame({
                    "trial": trial,
                    "seed": seed,
                    "n_shots": n_shots,
                    "method": method,
                    "query_image_id": image_ids[query_indices],
                    "true_person_id": true_labels,
                    "predicted_person_id": predictions,
                    "similarity": similarities,
                    "correct": predictions == true_labels,
                })
            )

fewshot_predictions_df = pd.concat(
    prediction_frames,
    ignore_index=True
)

fewshot_manifest_df = pd.DataFrame(manifest_rows)

fewshot_predictions_df.to_csv(
    REPORT_DIR / "arcface_fewshot_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

fewshot_manifest_df.to_csv(
    METADATA_DIR / "fewshot_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Eksperimen selesai dan hasil tersimpan.")

## Ringkasan few-shot

In [ ]:
trial_metric_rows = []

for keys, group in fewshot_predictions_df.groupby(
    ["n_shots", "method", "trial", "seed"]
):
    n_shots, method, trial, seed = keys

    trial_metric_rows.append({
        "n_shots": n_shots,
        "method": method,
        "trial": trial,
        "seed": seed,
        "accuracy": accuracy_score(
            group["true_person_id"],
            group["predicted_person_id"]
        ),
        "macro_f1": f1_score(
            group["true_person_id"],
            group["predicted_person_id"],
            labels=classes,
            average="macro",
            zero_division=0
        ),
    })

trial_metrics_df = pd.DataFrame(trial_metric_rows)

fewshot_summary_df = (
    trial_metrics_df
    .groupby(["n_shots", "method"])
    .agg(
        n_trials=("trial", "count"),
        mean_accuracy=("accuracy", "mean"),
        std_accuracy=("accuracy", "std"),
        min_accuracy=("accuracy", "min"),
        max_accuracy=("accuracy", "max"),
        mean_macro_f1=("macro_f1", "mean"),
    )
    .reset_index()
)

trial_metrics_df.to_csv(
    REPORT_DIR / "arcface_fewshot_trial_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

fewshot_summary_df.to_csv(
    REPORT_DIR / "arcface_fewshot_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

display(fewshot_summary_df.round(4))

## Encoder dan cache robustness

In [ ]:
import numpy as np
import pandas as pd

from PIL import Image, ImageEnhance, ImageFilter
from tqdm.auto import tqdm
from insightface import model_zoo

with np.load(
    ARTIFACTS_DIR / "arcface_embeddings.npz",
    allow_pickle=False
) as saved:
    base_embeddings = saved["embeddings"]
    base_image_ids = saved["image_ids"]
    base_person_ids = saved["person_ids"]

robust_metadata = (
    pd.read_csv(
        ARTIFACTS_DIR / "arcface_embedding_metadata.csv"
    )
    .set_index("image_id")
    .loc[base_image_ids]
)

if not np.array_equal(
    robust_metadata["person_id"].to_numpy(),
    base_person_ids
):
    raise ValueError("Urutan label metadata tidak sesuai.")

MODEL_PATH = (
    PROJECT_ROOT / "weights" / "insightface"
    / "models" / "buffalo_l" / "w600k_r50.onnx"
)

if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Model tidak ditemukan: {MODEL_PATH}")

encoder = model_zoo.get_model(
    str(MODEL_PATH),
    providers=["CPUExecutionProvider"]
)
encoder.prepare(ctx_id=-1)

ROBUST_CACHE_DIR = ARTIFACTS_DIR / "robustness_embeddings"
ROBUST_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Foto:", len(base_image_ids))
print("Encoder siap.")

## Embedding lima kondisi

In [ ]:
transforms = {
    "normal": lambda img: img,
    "gelap_0.5": lambda img: (
        ImageEnhance.Brightness(img).enhance(0.5)
    ),
    "terang_1.5": lambda img: (
        ImageEnhance.Brightness(img).enhance(1.5)
    ),
    "blur_radius_1": lambda img: (
        img.filter(ImageFilter.GaussianBlur(radius=1))
    ),
    "blur_radius_2": lambda img: (
        img.filter(ImageFilter.GaussianBlur(radius=2))
    ),
}

cache_summary_rows = []

for condition, transform in transforms.items():
    if condition == "normal":
        condition_embeddings = base_embeddings.copy()

    else:
        vectors = []

        for image_id in tqdm(
            base_image_ids,
            desc=condition,
            unit="foto"
        ):
            image_path = (
                PROJECT_ROOT
                / robust_metadata.loc[image_id, "aligned_path"]
            )

            with Image.open(image_path) as source:
                changed_image = transform(
                    source.convert("RGB")
                )

                if changed_image.size != tuple(encoder.input_size):
                    raise ValueError(
                        f"Ukuran gambar tidak sesuai: {image_id}"
                    )

                image_bgr = np.asarray(
                    changed_image
                )[:, :, ::-1].copy()

            vector = np.asarray(
                encoder.get_feat(image_bgr),
                dtype=np.float32
            ).reshape(-1)

            length = np.linalg.norm(vector)

            if (
                vector.size != 512
                or not np.isfinite(vector).all()
                or length <= 0
            ):
                raise ValueError(
                    f"Embedding tidak valid: {condition}, {image_id}"
                )

            vectors.append(vector / length)

        condition_embeddings = np.stack(vectors).astype(
            np.float32
        )

    if condition_embeddings.shape != base_embeddings.shape:
        raise ValueError("Jumlah atau dimensi embedding tidak sesuai.")

    cache_path = ROBUST_CACHE_DIR / f"{condition}.npz"
    temporary_path = cache_path.with_suffix(".tmp")

    with temporary_path.open("wb") as file:
        np.savez_compressed(
            file,
            embeddings=condition_embeddings,
            image_ids=base_image_ids,
            person_ids=base_person_ids,
            condition=np.array(condition)
        )

    temporary_path.replace(cache_path)

    cache_summary_rows.append({
        "condition": condition,
        "n_images": len(condition_embeddings),
        "dimension": condition_embeddings.shape[1],
        "normalized": np.allclose(
            np.linalg.norm(condition_embeddings, axis=1),
            1.0,
            atol=1e-5
        ),
    })

    print(f"Tersimpan: {cache_path.name}")

cache_summary_df = pd.DataFrame(cache_summary_rows)
display(cache_summary_df)

## Verifikasi pembagian robustness

In [ ]:
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score

ROBUST_CACHE_DIR = ARTIFACTS_DIR / "robustness_embeddings"
REPORT_DIR = PROJECT_ROOT / "reports" / "experiments"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

conditions = [
    "normal",
    "gelap_0.5",
    "terang_1.5",
    "blur_radius_1",
    "blur_radius_2",
]

with np.load(
    ARTIFACTS_DIR / "arcface_embeddings.npz",
    allow_pickle=False
) as saved:
    base_X = saved["embeddings"]
    base_ids = saved["image_ids"]
    base_labels = saved["person_ids"]

if len(np.unique(base_ids)) != len(base_ids):
    raise ValueError("Ada image_id duplikat.")

id_lookup = {
    image_id: index
    for index, image_id in enumerate(base_ids)
}

all_classes = np.unique(base_labels)

protocol_df = pd.read_csv(
    METADATA_DIR / "fewshot_manifest.csv"
)

if protocol_df.duplicated(["trial", "person_id"]).any():
    raise ValueError("Manifest memiliki baris identitas duplikat.")

trial_specs = []

for (trial, seed), group in protocol_df.groupby(
    ["trial", "seed"]
):
    group = group.sort_values("person_id")
    labels = group["person_id"].to_numpy()

    if not np.array_equal(labels, all_classes):
        raise ValueError(f"Identitas trial {trial} tidak lengkap.")

    query_idx = np.array([
        id_lookup[value]
        for value in group["query_image_id"]
    ])

    gallery_idx = np.array([
        [
            id_lookup[row.gallery_image_id_1],
            id_lookup[row.gallery_image_id_2]
        ]
        for row in group.itertuples()
    ])

    if np.intersect1d(query_idx, gallery_idx.ravel()).size:
        raise ValueError("Query masuk ke galeri.")

    if np.unique(gallery_idx).size != len(labels) * 2:
        raise ValueError("Foto referensi galeri terduplikasi.")

    if (
        not np.array_equal(base_labels[query_idx], labels)
        or not np.all(base_labels[gallery_idx] == labels[:, None])
    ):
        raise ValueError("Label pembagian tidak sesuai.")

    trial_specs.append({
        "trial": int(trial),
        "seed": int(seed),
        "labels": labels,
        "query_idx": query_idx,
        "gallery_idx": gallery_idx,
    })

if len(trial_specs) != 30:
    raise ValueError("Diperlukan tepat 30 pembagian.")

print("Pembagian siap:", len(trial_specs))
print("Identitas per pembagian:", len(all_classes))

## Evaluasi robustness

In [ ]:
prediction_frames = []
trial_metric_rows = []

for condition in conditions:
    cache_path = ROBUST_CACHE_DIR / f"{condition}.npz"

    with np.load(cache_path, allow_pickle=False) as cached:
        query_X = cached["embeddings"]
        cached_ids = cached["image_ids"]
        cached_labels = cached["person_ids"]

    if (
        query_X.shape != base_X.shape
        or not np.array_equal(cached_ids, base_ids)
        or not np.array_equal(cached_labels, base_labels)
    ):
        raise ValueError(f"Cache tidak sesuai: {condition}")

    if (
        not np.isfinite(query_X).all()
        or not np.allclose(
            np.linalg.norm(query_X, axis=1),
            1.0,
            atol=1e-5
        )
    ):
        raise ValueError(f"Embedding tidak valid: {condition}")

    for spec in tqdm(
        trial_specs,
        desc=condition,
        unit="pembagian"
    ):
        labels = spec["labels"]
        query_idx = spec["query_idx"]

        # Query memakai kondisi yang sedang diuji.
        query_vectors = query_X[query_idx]

        for n_shots in [1, 2]:
            gallery_idx = spec["gallery_idx"][:, :n_shots]

            # Galeri selalu memakai embedding normal.
            gallery_vectors = base_X[gallery_idx]

            flat_templates = gallery_vectors.reshape(
                -1, base_X.shape[1]
            )

            template_scores = (
                query_vectors @ flat_templates.T
            ).reshape(
                len(query_idx),
                len(labels),
                n_shots
            ).max(axis=2)

            centroids = gallery_vectors.mean(axis=1)
            lengths = np.linalg.norm(
                centroids, axis=1, keepdims=True
            )

            if np.any(lengths <= 1e-12):
                raise ValueError("Centroid memiliki panjang nol.")

            centroids = centroids / lengths
            centroid_scores = query_vectors @ centroids.T

            for method, scores in [
                ("nearest_template", template_scores),
                ("centroid", centroid_scores)
            ]:
                best_indices = np.argmax(scores, axis=1)
                predicted = labels[best_indices]

                best_scores = scores[
                    np.arange(len(query_idx)),
                    best_indices
                ]

                true_scores = scores[
                    np.arange(len(query_idx)),
                    np.arange(len(query_idx))
                ]

                prediction_frames.append(
                    pd.DataFrame({
                        "condition": condition,
                        "trial": spec["trial"],
                        "seed": spec["seed"],
                        "n_shots": n_shots,
                        "method": method,
                        "query_image_id": base_ids[query_idx],
                        "true_person_id": labels,
                        "predicted_person_id": predicted,
                        "similarity": best_scores,
                        "true_identity_similarity": true_scores,
                        "correct": predicted == labels,
                    })
                )

                trial_metric_rows.append({
                    "condition": condition,
                    "trial": spec["trial"],
                    "seed": spec["seed"],
                    "n_shots": n_shots,
                    "method": method,
                    "n_queries": len(query_idx),
                    "accuracy": accuracy_score(labels, predicted),
                    "macro_f1": f1_score(
                        labels,
                        predicted,
                        labels=all_classes,
                        average="macro",
                        zero_division=0
                    ),
                })

robustness_all_df = pd.concat(
    prediction_frames,
    ignore_index=True
)

robustness_trials_df = pd.DataFrame(trial_metric_rows)

robustness_all_df.to_csv(
    REPORT_DIR / "arcface_robustness_30trials_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

robustness_trials_df.to_csv(
    REPORT_DIR / "arcface_robustness_30trials_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Evaluasi selesai.")
print("Jumlah baris prediksi:", len(robustness_all_df))

## Ringkasan robustness

In [ ]:
robustness_summary_df = (
    robustness_trials_df
    .groupby(["condition", "n_shots", "method"])
    .agg(
        n_trials=("trial", "count"),
        mean_accuracy=("accuracy", "mean"),
        std_accuracy=("accuracy", "std"),
        min_accuracy=("accuracy", "min"),
        max_accuracy=("accuracy", "max"),
        mean_macro_f1=("macro_f1", "mean"),
    )
    .reset_index()
)

if not robustness_summary_df["n_trials"].eq(30).all():
    raise ValueError("Ada konfigurasi yang belum lengkap 30 percobaan.")

robustness_summary_df.to_csv(
    REPORT_DIR / "arcface_robustness_30trials_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

accuracy_table = robustness_summary_df.pivot(
    index="condition",
    columns=["n_shots", "method"],
    values="mean_accuracy"
).reindex(conditions)

print("Rata-rata akurasi:")
display(accuracy_table.round(4))

print("Detail kondisi blur radius 2:")
display(
    robustness_summary_df.loc[
        robustness_summary_df["condition"].eq("blur_radius_2")
    ].round(4)
)

## Perbandingan satu dan dua referensi

In [ ]:
paired_df = robustness_trials_df.pivot(
    index=["condition", "method", "trial", "seed"],
    columns="n_shots",
    values="accuracy"
).reset_index()

paired_df["delta"] = paired_df[2] - paired_df[1]

paired_df["two_shot_better"] = paired_df["delta"] > 1e-12
paired_df["two_shot_worse"] = paired_df["delta"] < -1e-12
paired_df["equal"] = paired_df["delta"].abs() <= 1e-12

benefit_df = (
    paired_df.groupby(["condition", "method"])
    .agg(
        better_trials=("two_shot_better", "sum"),
        equal_trials=("equal", "sum"),
        worse_trials=("two_shot_worse", "sum"),
        mean_delta=("delta", "mean"),
    )
    .reset_index()
)

benefit_df["mean_gain_percentage_points"] = (
    benefit_df.pop("mean_delta") * 100
)

benefit_df.to_csv(
    REPORT_DIR / "arcface_robustness_2shot_benefit.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    benefit_df.loc[
        benefit_df["condition"].eq("blur_radius_2")
    ].round(4)
)